In [ ]:
import arcpy
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import re

In [ ]:
# local timezone for Howard County, MD
timezone = 'America/New_York'  

gmt_format = '%m/%d/%Y %I:%M:%S %p'

In [ ]:
def calculate_date_range(monthyear, timezone):
    """
    This function calculates the start and end timestamps for a given month.

    Parameters
    ----------
    monthyear : str
        Month and year in the format 'MM/YYYY'.
    timezone : str
        Timezone for localization.

    Returns
    -------
    tuple
        (month_start, date_range_end)
        month_start : datetime
            Start of the given month with timezone localization.
        date_range_end : datetime
            End of the given month.
    """
    month_start = pd.to_datetime(monthyear, format='%m/%Y').tz_localize(timezone)
    next_month = (month_start + pd.DateOffset(months=1)).replace(day=1)
    date_range_end = next_month - timedelta(seconds=1)

    return month_start, date_range_end

def crosses_utc_day_boundary(prev_datetime, current_datetime):
    """
    This function checks if two timestamps fall on different UTC days.

    Parameters
    ----------
    prev_datetime : datetime
        The previous timestamp.
    current_datetime : datetime
        The current timestamp.

    Returns
    -------
    bool
        True if the timestamps cross a UTC day boundary, otherwise False.
    """
    return prev_datetime.date() != current_datetime.date()

def count_days_within_range(prev_datetime, current_datetime, daterange_start,
                            daterange_end):
    """
    This functions counts the number of days between two timestamps,
    considering month boundaries.

    Parameters
    ----------
    prev_datetime : datetime
        The previous timestamp.
    current_datetime : datetime
        The current timestamp.
    daterange_start : datetime
        Start of the target month in local time.
    daterange_end : datetime
        End of the target month in local time.

    Returns
    -------
    dict
        A dictionary with month-year keys ('MM/YYYY') and the number of counted 
        days as values.
    """
    day_counts = {} 
    
    if (prev_datetime.month == current_datetime.month and 
        prev_datetime.year == current_datetime.year):
        day_start = (prev_datetime.replace(hour=0, minute=0, second=0, 
                                           microsecond=0) + timedelta(days=1))
        
        current_month_year = daterange_start.strftime('%m/%Y')
        
        day_counts[current_month_year] = 0

        while day_start <= current_datetime:
            day_start_local = day_start.tz_convert(timezone)

            if (daterange_start <= day_start_local <= daterange_end):
                day_counts[current_month_year] += 1

            day_start += timedelta(days=1)

    else:
        previous_month_end = daterange_start - timedelta(microseconds=1)
        
        day_start = (prev_datetime.replace(hour=0, minute=0, second=0, 
                                           microsecond=0) + timedelta(days=1))
        
        while day_start <= current_datetime:
            day_start_local = day_start.tz_convert(timezone)

            if day_start_local <= previous_month_end: 
                previous_month_year = previous_month_end.strftime('%m/%Y')
                if previous_month_year not in day_counts:
                    day_counts[previous_month_year] = 0
                day_counts[previous_month_year] += 1

            elif daterange_start <= day_start_local <= daterange_end:
                current_month_year = daterange_start.strftime('%m/%Y')
                if current_month_year not in day_counts:
                    day_counts[current_month_year] = 0
                day_counts[current_month_year] += 1

            day_start += timedelta(days=1)

    return day_counts

def is_more_than_equal_2_hours(prev_datetime, current_datetime):
    """
    This function checks if the time difference between two datetimes is 
    greater than or equal to 2 hours.

    Parameters
    ----------
    prev_datetime : datetime
        The earlier timestamp.
    current_datetime : datetime
        The later timestamp.

    Returns
    -------
    bool
        True if the time difference is >= 2 hours, otherwise False.
    """
    time_difference = current_datetime - prev_datetime
    
    return time_difference >= timedelta(hours=2)

def create_hour_count_array(group):
    """
    This function creates a list of 24 hourly counts from a grouped DataFrame.

    Parameters
    ----------
    group : pandas.DataFrame
        A DataFrame group containing 'hour' and 'pophuman' columns.
        Each row represents a count for a specific hour.

    Returns
    -------
    list
        A list of 24 integers where each index corresponds to the
        count for that hour (0 to 23). 
    """
    hour_counts = [0] * 24
    
    for hour, count in zip(group['hour'], group['pophuman']):
        hour_counts[hour] = count
    
    return hour_counts

def format_array(arr):
    """
    This functions converts a list to a string and removes all spaces.

    Parameters
    ----------
    arr : list
        A list of elements.

    Returns
    -------
    str
        A string representation of the list with no spaces.
    """
    return str(arr).replace(" ", "")

### Deer Visit Counts

In [ ]:
arcpy.analysis.SpatialJoin(
    target_features="deer_GPS_2018_2020",
    join_features="hexagonal_tessellation",
    out_feature_class=r"path\to\your\ArcGISProgeodatabase.gdb\deer_GPS_SpatialJoin",
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    field_mapping='event_id "event_id" true true false 8 Double 0 0,First,#,deer_GPS_2018_2020,event_id,-1,-1;date_local "date_local" true true false 8 Date 0 0,First,#,deer_GPS_2018_2020,date_local,-1,-1;loc_long "loc_long" true true false 8 Double 0 0,First,#,deer_GPS_2018_2020,loc_long,-1,-1;loc_lat "loc_lat" true true false 8 Double 0 0,First,#,deer_GPS_2018_2020,loc_lat,-1,-1;tag_loc_id "tag_loc_id" true true false 8000 Text 0 0,First,#,deer_GPS_2018_2020,tag_loc_id,0,7999;ind_loc_id "ind_loc_id" true true false 4 Long 0 0,First,#,deer_GPS_2018_2020,ind_loc_id,-1,-1;gmt_dt "gmt_dt" true true false 8000 Text 0 0,First,#,deer_GPS_2018_2020,gmt_dt,0,7999;GRID_ID "GRID_ID" true true false 12 Text 0 0,First,#,hexagonal_tessellation,GRID_ID,0,11;Shape_Length "Shape_Length" false true true 8 Double 0 0,First,#,hexagonal_tessellation,Shape_Length,-1,-1;Shape_Area "Shape_Area" false true true 8 Double 0 0,First,#,hexagonal_tessellation,Shape_Area,-1,-1',
    match_option="INTERSECT",
    search_radius=None,
    distance_field_name="",
    match_fields=None
)

In [ ]:
fields_list = [
    field.name for field in arcpy.ListFields('deer_GPS_SpatialJoin')
]

In [ ]:
fields = ['ind_loc_id', 'gmt_dt', 'GRID_ID']

data = pd.DataFrame.from_records(
    data=arcpy.da.SearchCursor("deer_GPS_SpatialJoin", fields),
    columns=fields
)

In [ ]:
data['gmt_dt'] = pd.to_datetime(data['gmt_dt'], format=gmt_format)  

data['gmt_dt'] = data['gmt_dt'].dt.tz_localize('UTC')  
data['dt_local'] = data['gmt_dt'].dt.tz_convert(timezone)

In [ ]:
# Note: deer data have to be sorted by ind_loc_id and gmt_dt
data.sort_values(
    by=['ind_loc_id', 'gmt_dt'], ascending=[True, True], inplace=True
)

In [ ]:
data['month_year'] = data['dt_local'].dt.strftime('%m/%Y') 
data['month_year'] = data['month_year'].astype('string')

In [ ]:
data['GRID_ID'] = data['GRID_ID'].astype('string')

In [ ]:
# create a new dataframe to store the deer visit counts of
# each GRID_ID that has deer
count_df = pd.DataFrame(columns=['GRID_ID'])

grid_ids = data['GRID_ID'].unique()
count_df['GRID_ID'] = grid_ids

In [ ]:
months = data['month_year'].unique()

for month in months:
    count_df[month] = np.nan

In [ ]:
previous_gid = None

In [ ]:
for index, row in data.iterrows():
    gid = row['GRID_ID']
    month_year = row['month_year']  
    deer_id = row['ind_loc_id']
    datetime_utc = row['gmt_dt']
    datetime_local = row['dt_local']
        
    date_range_start, date_range_end = calculate_date_range(month_year, timezone)

    if (previous_gid != gid and 
        date_range_start <= datetime_local <= date_range_end):
        count_df.loc[count_df['GRID_ID'] == gid, month_year] = (
            count_df.loc[count_df['GRID_ID'] == gid, month_year]
            .fillna(0) + 1
        )
    
    elif (previous_gid == gid and 
          previous_deer_id != deer_id and 
          date_range_start <= datetime_local <= date_range_end):
        count_df.loc[count_df['GRID_ID'] == gid, month_year] = (
            count_df.loc[count_df['GRID_ID'] == gid, month_year]
            .fillna(0) + 1
        )
        
    elif (crosses_utc_day_boundary(previous_datetime_utc, datetime_utc) and 
          previous_gid == gid and 
          previous_deer_id == deer_id):

        day_counts = count_days_within_range(
            previous_datetime_utc, datetime_utc, date_range_start, date_range_end
        )

        for month_year, count in day_counts.items():
            count_df.loc[count_df['GRID_ID'] == gid, month_year] = (
                count_df.loc[count_df['GRID_ID'] == gid, month_year]
                .fillna(0) + count
            )
          
    previous_deer_id = deer_id
    previous_datetime_utc = datetime_utc
    previous_gid = gid

In [ ]:
# rename month_year (e.g., 01/2018) columns to 'Jan18_deer', ...
count_df.rename(
    columns=lambda c_name: (
        pd.to_datetime(c_name, format='%m/%Y').strftime('%b%y') + "_deer"
        if '/' in c_name else c_name
    ),
    inplace=True
)

In [ ]:
count_df.iloc[:, 1:] = count_df.iloc[:, 1:].fillna(0).astype('int64')

count_dict = count_df.set_index('GRID_ID').to_dict(orient='index')

In [ ]:
feature_class = r"path\to\your\ArcGISProgeodatabase.gdb\hexagonal_tessellation"

# check if fields exist, and add them if they don't; adds deer fields
for c_field in count_df.columns[1:]: 
    if c_field not in fields_list:
        arcpy.AddField_management(feature_class, c_field, 'LONG')

# update the feature class with the deer visit counts
with arcpy.da.UpdateCursor(
    feature_class, ['GRID_ID'] + list(count_df.columns[1:])
) as cursor:
    for row in cursor:
        gridid = row[0]  
        if gridid in count_dict:
            for i, field in enumerate(count_df.columns[1:], start=1):
                row[i] = count_dict[gridid][field]
            cursor.updateRow(row)

In [ ]:
# replace <Null> values with 0 without overwriting existing values
for c_field in count_df.columns[1:]:  
    arcpy.CalculateField_management(
        feature_class, c_field,
        "0 if !{}! is None else !{}!".format(c_field, c_field),
        "PYTHON3"
    )

### Human Commercial Visit Counts

Human foot traffic data (Safegraph Places and Advan Monthly Patterns) were collected from Dewey Data.

In [ ]:
arcpy.analysis.SummarizeWithin(
    in_polygons="hexagonal_tessellation",
    in_sum_features="poi_human_2018_2020",
    out_feature_class=r"path\to\your\ArcGISProgeodatabase.gdb\hexagonal_tessellation_SumWithin",
    keep_all_polygons="KEEP_ALL",
    sum_fields="Jan18_hum Sum;Feb18_hum Sum;Mar18_hum Sum;Apr18_hum Sum;May18_hum Sum;Jun18_hum Sum;Jul18_hum Sum;Aug18_hum Sum;Sep18_hum Sum;Oct18_hum Sum;Nov18_hum Sum;Dec18_hum Sum;Jan19_hum Sum;Feb19_hum Sum;Mar19_hum Sum;Apr19_hum Sum;May19_hum Sum;Jun19_hum Sum;Jul19_hum Sum;Aug19_hum Sum;Sep19_hum Sum;Oct19_hum Sum;Nov19_hum Sum;Dec19_hum Sum;Jan20_hum Sum",
    sum_shape="ADD_SHAPE_SUM",
    shape_unit="SQUAREMETERS",
    group_field=None,
    add_min_maj="NO_MIN_MAJ",
    add_group_percent="NO_PERCENT",
    out_group_table=None
)

In [ ]:
feature_class2 = r"path\to\your\ArcGISProgeodatabase.gdb\hexagonal_tessellation_SumWithin"

fields_dict = {
    field.name: field.aliasName 
    for field in arcpy.ListFields(feature_class2)
}

In [ ]:
# dictionary to store renaming mappings for field names and alias names
rename_dict = {}

# regex pattern to match field names like "sum_Jan18_hum"
field_pattern = re.compile(
    r"sum_(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)(\d{2})_hum"
)

In [ ]:
# loop through fields and apply renaming if they match the pattern
for field, alias in fields_dict.items():
    match = field_pattern.match(field)
    if match:
        month, year = match.groups()
        new_name = f"{month}{year}_hum"  
        new_alias = f"{month}{year}_hum"
        rename_dict[field] = (new_name, new_alias)

if 'Point_Count' in fields_dict:
    rename_dict['Point_Count'] = ('POI_count', 'POI_count') 

In [ ]:
# rename fields and update alias names
for old_name, (new_name, new_alias) in rename_dict.items():
    try:
        arcpy.management.AlterField(
            feature_class2, old_name, new_name, new_field_alias=new_alias
        )
        print(
            f"Renamed '{old_name}' → '{new_name}', alias updated to '{new_alias}'"
        )
    except Exception as e:
        print(f"Unable to rename '{old_name}': {e}")

*Note*: Human commercial visit counts for the grid cell containing only Rockburn Branch Park were identified as outliers for the period August–October 2019. These values were replaced with corresponding data from August–October 2018.

### Residential and Commercial Building Area

The building footprints (`Buildings_Major`) and land use (`Land Use`) shapefiles are available here: https://data.howardcountymd.gov/

We reclassified land use categories as described in the paper, including residential and commercial areas, among others. Then, we did a spatial join between the building footprints and the land use shapefiles. 

In [ ]:
arcpy.analysis.PairwiseIntersect(
    in_features="hexagonal_tessellation;buildings_landuse",
    out_feature_class=r"path\to\your\ArcGISProgeodatabase.gdb\hexagonal_tess_PairwiseInter",
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="INPUT"
)

In [ ]:
arcpy.conversion.ExportFeatures(
    in_features="hexagonal_tess_PairwiseInter",
    out_features=r"path\to\your\ArcGISProgeodatabase.gdb\hexagonal_tess_PairwiseInter_res",
    where_clause="Build_Use = 'Residential'",
    use_field_alias_as_name="NOT_USE_ALIAS",
    field_mapping='FID_hexagonal_tessellation "FID_hexagonal_tessellation" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,FID_hexagonal_tessellation,-1,-1;GRID_ID "GRID_ID" true true false 12 Text 0 0,First,#,hexagonal_tess_PairwiseInter,GRID_ID,0,11;Jan18_deer "Jan18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Jan18_deer,-1,-1;Feb18_deer "Feb18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Feb18_deer,-1,-1;Mar18_deer "Mar18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Mar18_deer,-1,-1;Apr18_deer "Apr18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Apr18_deer,-1,-1;May18_deer "May18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,May18_deer,-1,-1;Jun18_deer "Jun18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Jun18_deer,-1,-1;Jul18_deer "Jul18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Jul18_deer,-1,-1;Aug18_deer "Aug18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Aug18_deer,-1,-1;Sep18_deer "Sep18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Sep18_deer,-1,-1;Oct18_deer "Oct18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Oct18_deer,-1,-1;Nov18_deer "Nov18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Nov18_deer,-1,-1;Dec18_deer "Dec18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Dec18_deer,-1,-1;Jan19_deer "Jan19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Jan19_deer,-1,-1;Feb19_deer "Feb19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Feb19_deer,-1,-1;Mar19_deer "Mar19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Mar19_deer,-1,-1;Apr19_deer "Apr19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Apr19_deer,-1,-1;May19_deer "May19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,May19_deer,-1,-1;Jun19_deer "Jun19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Jun19_deer,-1,-1;Jul19_deer "Jul19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Jul19_deer,-1,-1;Aug19_deer "Aug19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Aug19_deer,-1,-1;Sep19_deer "Sep19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Sep19_deer,-1,-1;Oct19_deer "Oct19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Oct19_deer,-1,-1;Nov19_deer "Nov19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Nov19_deer,-1,-1;Dec19_deer "Dec19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Dec19_deer,-1,-1;Jan20_deer "Jan20_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Jan20_deer,-1,-1;FID_buildings_landuse "FID_buildings_landuse" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,FID_buildings_landuse,-1,-1;CAPYEAR "CAPYEAR" true true false 8 Double 0 0,First,#,hexagonal_tess_PairwiseInter,CAPYEAR,-1,-1;FEATURE "FEATURE" true true false 50 Text 0 0,First,#,hexagonal_tess_PairwiseInter,FEATURE,0,49;id "id" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,id,-1,-1;Generalize "Generalize" true true false 254 Text 0 0,First,#,hexagonal_tess_PairwiseInter,Generalize,0,253;Detailed_U "Detailed_U" true true false 254 Text 0 0,First,#,hexagonal_tess_PairwiseInter,Detailed_U,0,253;Use_Code "Use_Code" true true false 10 Text 0 0,First,#,hexagonal_tess_PairwiseInter,Use_Code,0,9;Build_Use "Build_Use" true true false 254 Text 0 0,First,#,hexagonal_tess_PairwiseInter,Build_Use,0,253;Shape_Leng "Shape_Leng" true true false 8 Double 0 0,First,#,hexagonal_tess_PairwiseInter,Shape_Leng,-1,-1;Shape_Length "Shape_Length" false true true 8 Double 0 0,First,#,hexagonal_tess_PairwiseInter,Shape_Length,-1,-1;Shape_Area "Shape_Area" false true true 8 Double 0 0,First,#,hexagonal_tess_PairwiseInter,Shape_Area,-1,-1',
    sort_field=None
)

In [ ]:
arcpy.conversion.ExportFeatures(
    in_features="hexagonal_tess_PairwiseInter",
    out_features=r"path\to\your\ArcGISProgeodatabase.gdb\hexagonal_tess_PairwiseInter_com",
    where_clause="Build_Use = 'Commercial'",
    use_field_alias_as_name="NOT_USE_ALIAS",
    field_mapping='FID_hexagonal_tessellation "FID_hexagonal_tessellation" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,FID_hexagonal_tessellation,-1,-1;GRID_ID "GRID_ID" true true false 12 Text 0 0,First,#,hexagonal_tess_PairwiseInter,GRID_ID,0,11;Jan18_deer "Jan18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Jan18_deer,-1,-1;Feb18_deer "Feb18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Feb18_deer,-1,-1;Mar18_deer "Mar18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Mar18_deer,-1,-1;Apr18_deer "Apr18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Apr18_deer,-1,-1;May18_deer "May18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,May18_deer,-1,-1;Jun18_deer "Jun18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Jun18_deer,-1,-1;Jul18_deer "Jul18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Jul18_deer,-1,-1;Aug18_deer "Aug18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Aug18_deer,-1,-1;Sep18_deer "Sep18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Sep18_deer,-1,-1;Oct18_deer "Oct18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Oct18_deer,-1,-1;Nov18_deer "Nov18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Nov18_deer,-1,-1;Dec18_deer "Dec18_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Dec18_deer,-1,-1;Jan19_deer "Jan19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Jan19_deer,-1,-1;Feb19_deer "Feb19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Feb19_deer,-1,-1;Mar19_deer "Mar19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Mar19_deer,-1,-1;Apr19_deer "Apr19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Apr19_deer,-1,-1;May19_deer "May19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,May19_deer,-1,-1;Jun19_deer "Jun19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Jun19_deer,-1,-1;Jul19_deer "Jul19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Jul19_deer,-1,-1;Aug19_deer "Aug19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Aug19_deer,-1,-1;Sep19_deer "Sep19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Sep19_deer,-1,-1;Oct19_deer "Oct19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Oct19_deer,-1,-1;Nov19_deer "Nov19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Nov19_deer,-1,-1;Dec19_deer "Dec19_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Dec19_deer,-1,-1;Jan20_deer "Jan20_deer" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,Jan20_deer,-1,-1;FID_buildings_landuse "FID_buildings_landuse" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,FID_buildings_landuse,-1,-1;CAPYEAR "CAPYEAR" true true false 8 Double 0 0,First,#,hexagonal_tess_PairwiseInter,CAPYEAR,-1,-1;FEATURE "FEATURE" true true false 50 Text 0 0,First,#,hexagonal_tess_PairwiseInter,FEATURE,0,49;id "id" true true false 4 Long 0 0,First,#,hexagonal_tess_PairwiseInter,id,-1,-1;Generalize "Generalize" true true false 254 Text 0 0,First,#,hexagonal_tess_PairwiseInter,Generalize,0,253;Detailed_U "Detailed_U" true true false 254 Text 0 0,First,#,hexagonal_tess_PairwiseInter,Detailed_U,0,253;Use_Code "Use_Code" true true false 10 Text 0 0,First,#,hexagonal_tess_PairwiseInter,Use_Code,0,9;Build_Use "Build_Use" true true false 254 Text 0 0,First,#,hexagonal_tess_PairwiseInter,Build_Use,0,253;Shape_Leng "Shape_Leng" true true false 8 Double 0 0,First,#,hexagonal_tess_PairwiseInter,Shape_Leng,-1,-1;Shape_Length "Shape_Length" false true true 8 Double 0 0,First,#,hexagonal_tess_PairwiseInter,Shape_Length,-1,-1;Shape_Area "Shape_Area" false true true 8 Double 0 0,First,#,hexagonal_tess_PairwiseInter,Shape_Area,-1,-1',
    sort_field=None
)

In [ ]:
arcpy.analysis.SummarizeWithin(
    in_polygons="hexagonal_tessellation_SumWithin",
    in_sum_features="hexagonal_tess_PairwiseInter_res",
    out_feature_class=r"path\to\your\ArcGISProgeodatabase.gdb\hexagonal_tessellation_SumWithin_res",
    keep_all_polygons="KEEP_ALL",
    sum_fields="Shape_Area Sum",
    sum_shape="ADD_SHAPE_SUM",
    shape_unit="SQUAREMETERS",
    group_field=None,
    add_min_maj="NO_MIN_MAJ",
    add_group_percent="NO_PERCENT",
    out_group_table=None
)

In [ ]:
arcpy.management.AlterField(
    in_table="hexagonal_tessellation_SumWithin_res",
    field="sum_Area_SQUAREMETERS",
    new_field_name="res_m2",
    new_field_alias="res_m2",
    field_type="",
    field_length=8,
    field_is_nullable="NULLABLE",
    clear_field_alias="DO_NOT_CLEAR"
)

In [ ]:
arcpy.analysis.SummarizeWithin(
    in_polygons="hexagonal_tessellation_SumWithin_res",
    in_sum_features="hexagonal_tess_PairwiseInter_com",
    out_feature_class=r"path\to\your\ArcGISProgeodatabase.gdb\hexagonal_tessellation_SumWithin_rescom",
    keep_all_polygons="KEEP_ALL",
    sum_fields="Shape_Area Sum",
    sum_shape="ADD_SHAPE_SUM",
    shape_unit="SQUAREMETERS",
    group_field=None,
    add_min_maj="NO_MIN_MAJ",
    add_group_percent="NO_PERCENT",
    out_group_table=None
)

In [ ]:
arcpy.management.AlterField(
    in_table="hexagonal_tessellation_SumWithin_rescom",
    field="sum_Area_SQUAREMETERS",
    new_field_name="com_m2",
    new_field_alias="com_m2",
    field_type="",
    field_length=8,
    field_is_nullable="NULLABLE",
    clear_field_alias="DO_NOT_CLEAR"
)

In [ ]:
arcpy.management.DeleteField(
    in_table="hexagonal_tessellation_SumWithin_rescom",
    drop_field="sum_Shape_Area;Polygon_Count;sum_Shape_Area_1;Polygon_Count_1",
    method="DELETE_FIELDS"
)

### Estimated Nighttime Population

Data are available here: https://hub.worldpop.org/geodata/summary?id=49727

In [ ]:
arcpy.ia.ZonalStatisticsAsTable(
    in_zone_data="hexagonal_tessellation_SumWithin_rescom",
    zone_field="GRID_ID",
    in_value_raster="worldpop",
    out_table=r"path\to\your\ArcGISProgeodatabase.gdb\ZonalSt_hextab",
    ignore_nodata="DATA",
    statistics_type="SUM",
    process_as_multidimensional="CURRENT_SLICE",
    percentile_values=90,
    percentile_interpolation_type="AUTO_DETECT",
    circular_calculation="ARITHMETIC",
    circular_wrap_value=360,
    out_join_layer="ZonalSt_hex_fc"
)

In [ ]:
arcpy.conversion.ExportFeatures(
    in_features="ZonalSt_hex_fc",
    out_features=r"path\to\your\ArcGISProgeodatabase.gdb\ZonalSt_hex_fc_ExportFeature",
    where_clause="",
    use_field_alias_as_name="NOT_USE_ALIAS",
    field_mapping='GRID_ID "GRID_ID" true true false 12 Text 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.GRID_ID,0,11;Jan18_deer "Jan18_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Jan18_deer,-1,-1;Feb18_deer "Feb18_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Feb18_deer,-1,-1;Mar18_deer "Mar18_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Mar18_deer,-1,-1;Apr18_deer "Apr18_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Apr18_deer,-1,-1;May18_deer "May18_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.May18_deer,-1,-1;Jun18_deer "Jun18_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Jun18_deer,-1,-1;Jul18_deer "Jul18_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Jul18_deer,-1,-1;Aug18_deer "Aug18_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Aug18_deer,-1,-1;Sep18_deer "Sep18_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Sep18_deer,-1,-1;Oct18_deer "Oct18_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Oct18_deer,-1,-1;Nov18_deer "Nov18_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Nov18_deer,-1,-1;Dec18_deer "Dec18_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Dec18_deer,-1,-1;Jan19_deer "Jan19_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Jan19_deer,-1,-1;Feb19_deer "Feb19_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Feb19_deer,-1,-1;Mar19_deer "Mar19_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Mar19_deer,-1,-1;Apr19_deer "Apr19_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Apr19_deer,-1,-1;May19_deer "May19_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.May19_deer,-1,-1;Jun19_deer "Jun19_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Jun19_deer,-1,-1;Jul19_deer "Jul19_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Jul19_deer,-1,-1;Aug19_deer "Aug19_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Aug19_deer,-1,-1;Sep19_deer "Sep19_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Sep19_deer,-1,-1;Oct19_deer "Oct19_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Oct19_deer,-1,-1;Nov19_deer "Nov19_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Nov19_deer,-1,-1;Dec19_deer "Dec19_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Dec19_deer,-1,-1;Jan20_deer "Jan20_deer" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Jan20_deer,-1,-1;Jan18_hum "Jan18_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Jan18_hum,-1,-1;Feb18_hum "Feb18_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Feb18_hum,-1,-1;Mar18_hum "Mar18_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Mar18_hum,-1,-1;Apr18_hum "Apr18_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Apr18_hum,-1,-1;May18_hum "May18_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.May18_hum,-1,-1;Jun18_hum "Jun18_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Jun18_hum,-1,-1;Jul18_hum "Jul18_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Jul18_hum,-1,-1;Aug18_hum "Aug18_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Aug18_hum,-1,-1;Sep18_hum "Sep18_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Sep18_hum,-1,-1;Oct18_hum "Oct18_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Oct18_hum,-1,-1;Nov18_hum "Nov18_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Nov18_hum,-1,-1;Dec18_hum "Dec18_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Dec18_hum,-1,-1;Jan19_hum "Jan19_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Jan19_hum,-1,-1;Feb19_hum "Feb19_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Feb19_hum,-1,-1;Mar19_hum "Mar19_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Mar19_hum,-1,-1;Apr19_hum "Apr19_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Apr19_hum,-1,-1;May19_hum "May19_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.May19_hum,-1,-1;Jun19_hum "Jun19_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Jun19_hum,-1,-1;Jul19_hum "Jul19_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Jul19_hum,-1,-1;Aug19_hum "Aug19_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Aug19_hum,-1,-1;Sep19_hum "Sep19_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Sep19_hum,-1,-1;Oct19_hum "Oct19_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Oct19_hum,-1,-1;Nov19_hum "Nov19_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Nov19_hum,-1,-1;Dec19_hum "Dec19_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Dec19_hum,-1,-1;Jan20_hum "Jan20_hum" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Jan20_hum,-1,-1;POI_count "POI_count" true true false 4 Long 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.POI_count,-1,-1;res_m2 "res_m2" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.res_m2,-1,-1;Shape_Length "Shape_Length" false true true 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Shape_Length,-1,-1;Shape_Area "Shape_Area" false true true 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.Shape_Area,-1,-1;com_m2 "com_m2" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,hexagonal_tessellation_SumWithin_rescom.com_m2,-1,-1;pop_total "pop_total" true true false 8 Double 0 0,First,#,ZonalSt_hex_fc,ZonalSt_hextab.SUM,-1,-1',
    sort_field=None
)

In [ ]:
c_field = "pop_total"

arcpy.CalculateField_management(
    "ZonalSt_hex_fc_ExportFeature", c_field,
    "0 if !{}! is None else !{}!".format(c_field, c_field), "PYTHON3"
)

*Note*: Park hexagons in the `ZonalSt_hex_fc_ExportFeature` layer were merged in ArcGIS Pro, and `GRID_ID` was renamed `GRID_ID_p` to reflect the inclusion of merged park areas, such as "Park-1", etc. A hexagon was categorized as a park hexagon if its center fell within a Howard County park.

In [ ]:
arcpy.management.Dissolve(
    in_features="ZonalSt_hex_fc_ExportFeature",
    out_feature_class=r"path\to\your\ArcGISProgeodatabase.gdb\visit_counts_201801_202001",
    dissolve_field="GRID_ID_p",
    statistics_fields="Jan18_deer SUM;Feb18_deer SUM;Mar18_deer SUM;Apr18_deer SUM;May18_deer SUM;Jun18_deer SUM;Jul18_deer SUM;Aug18_deer SUM;Sep18_deer SUM;Oct18_deer SUM;Nov18_deer SUM;Dec18_deer SUM;Jan19_deer SUM;Feb19_deer SUM;Mar19_deer SUM;Apr19_deer SUM;May19_deer SUM;Jun19_deer SUM;Jul19_deer SUM;Aug19_deer SUM;Sep19_deer SUM;Oct19_deer SUM;Nov19_deer SUM;Dec19_deer SUM;Jan20_deer SUM;Jan18_hum SUM;Feb18_hum SUM;Mar18_hum SUM;Apr18_hum SUM;May18_hum SUM;Jun18_hum SUM;Jul18_hum SUM;Aug18_hum SUM;Sep18_hum SUM;Oct18_hum SUM;Nov18_hum SUM;Dec18_hum SUM;Jan19_hum SUM;Feb19_hum SUM;Mar19_hum SUM;Apr19_hum SUM;May19_hum SUM;Jun19_hum SUM;Jul19_hum SUM;Aug19_hum SUM;Sep19_hum SUM;Oct19_hum SUM;Nov19_hum SUM;Dec19_hum SUM;Jan20_hum SUM;POI_count SUM;res_m2 SUM;com_m2 SUM;pop_total SUM;GRID_ID_p COUNT",
    multi_part="MULTI_PART",
    unsplit_lines="DISSOLVE_LINES",
    concatenation_separator=""
)

In [ ]:
feature_class3 = r"path\to\your\ArcGISProgeodatabase.gdb\visit_counts_201801_202001"

fields_dict2 = {
    field.name: field.aliasName
    for field in arcpy.ListFields(feature_class3)
}

In [ ]:
# dictionary to store renaming mappings for field names and alias names
rename_dict2 = {}

# regex pattern to match field names starting with "SUM_"
sum_pattern = re.compile(r"^SUM_(.+)")

In [ ]:
# loop through fields and apply renaming if they match the pattern
for field in fields_dict2:
    match = sum_pattern.match(field)
    if match:
        rest = match.group(1)
        rename_dict2[field] = (rest, rest)

if 'GRID_ID_p' in fields_dict2:
    rename_dict2["GRID_ID_p"] = ('GRID_ID_p', 'GRID_ID_p') 

if 'COUNT_GRID_ID_p' in fields_dict2:
    rename_dict2['COUNT_GRID_ID_p'] = ('GRID_count', 'GRID_count')

In [ ]:
# rename fields and update alias names
# this is the dataset that is used in the models
for old_name, (new_name, new_alias) in rename_dict2.items():
    try:
        arcpy.management.AlterField(
            feature_class3, old_name, new_name, new_field_alias=new_alias
        )
        print(
            f"Renamed '{old_name}' → '{new_name}', alias updated to '{new_alias}'"
        )
    except Exception as e:
        print(f"Unable to rename '{old_name}': {e}")

In [ ]:
# create a new feature class of the tessellation that has parks merged
arcpy.conversion.ExportFeatures(
    in_features="visit_counts_201801_202001",
    out_features=r"path\to\your\ArcGISProgeodatabase.gdb\parksmerged_tessellation",
    where_clause="",
    use_field_alias_as_name="NOT_USE_ALIAS",
    field_mapping='GRID_ID_p "GRID_ID" true true false 12 Text 0 0,First,#,visit_counts_201801_202001,GRID_ID_p,0,11',
    sort_field=None
)

### Deer Popularity by Hour

In [ ]:
# make a copy of the deer data to be joined with the merged tessellation
arcpy.management.CopyFeatures(
    in_features="deer_GPS_2018_2020",
    out_feature_class=r"path\to\your\ArcGISProgeodatabase.gdb\deer_GPS_2018_2020_CopyFeatures",
    config_keyword="",
    spatial_grid_1=None,
    spatial_grid_2=None,
    spatial_grid_3=None
)

In [ ]:
arcpy.analysis.SpatialJoin(
    target_features="deer_GPS_2018_2020_CopyFeatures",
    join_features="parksmerged_tessellation",
    out_feature_class=r"path\to\your\ArcGISProgeodatabase.gdb\deer_GPS_SpatialJoin_2",
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    field_mapping='event_id "event_id" true true false 8 Double 0 0,First,#,deer_GPS_2018_2020_CopyFeatures,event_id,-1,-1;date_local "date_local" true true false 8 Date 0 0,First,#,deer_GPS_2018_2020_CopyFeatures,date_local,-1,-1;loc_long "loc_long" true true false 8 Double 0 0,First,#,deer_GPS_2018_2020_CopyFeatures,loc_long,-1,-1;loc_lat "loc_lat" true true false 8 Double 0 0,First,#,deer_GPS_2018_2020_CopyFeatures,loc_lat,-1,-1;tag_loc_id "tag_loc_id" true true false 8000 Text 0 0,First,#,deer_GPS_2018_2020_CopyFeatures,tag_loc_id,0,7999;ind_loc_id "ind_loc_id" true true false 4 Long 0 0,First,#,deer_GPS_2018_2020_CopyFeatures,ind_loc_id,-1,-1;gmt_dt "gmt_dt" true true false 8000 Text 0 0,First,#,deer_GPS_2018_2020_CopyFeatures,gmt_dt,0,7999;GRID_ID_p "GRID_ID" true true false 12 Text 0 0,First,#,parksmerged_tessellation,GRID_ID_p,0,11;Shape_Length "Shape_Length" false true true 8 Double 0 0,First,#,parksmerged_tessellation,Shape_Length,-1,-1;Shape_Area "Shape_Area" false true true 8 Double 0 0,First,#,parksmerged_tessellation,Shape_Area,-1,-1',
    match_option="INTERSECT",
    search_radius=None,
    distance_field_name="",
    match_fields=None
)

In [ ]:
fields_list_pop = [
    field.name for field in arcpy.ListFields('deer_GPS_SpatialJoin_2')
]

In [ ]:
fields_pop = ['ind_loc_id', 'gmt_dt', 'GRID_ID_p']

data_pop = pd.DataFrame.from_records(
    data=arcpy.da.SearchCursor("deer_GPS_SpatialJoin_2", fields_pop),
    columns=fields_pop
)

In [ ]:
data_pop['gmt_dt'] = pd.to_datetime(data_pop['gmt_dt'], format=gmt_format)  

data_pop['gmt_dt'] = data_pop['gmt_dt'].dt.tz_localize('UTC')  
data_pop['dt_local'] = data_pop['gmt_dt'].dt.tz_convert(timezone)

In [ ]:
# Note: deer data have to be sorted by ind_loc_id and gmt_dt
data_pop.sort_values(
    by=['ind_loc_id', 'gmt_dt'], ascending=[True, True], inplace=True
)

In [ ]:
data_pop['month_year'] = data_pop['dt_local'].dt.strftime('%m/%Y') 
data_pop['month_year'] = data_pop['month_year'].astype('string')

In [ ]:
data_pop['hour'] = data_pop['dt_local'].dt.hour

In [ ]:
data_pop['GRID_ID_p'] = data_pop['GRID_ID_p'].astype('string')

In [ ]:
# create a new dataframe to store the deer hourly popularity of
# each GRID_ID_p that has deer
count_df_pop = pd.DataFrame(columns=['GRID_ID_p'])

grid_ids_pop = data_pop['GRID_ID_p'].unique()
count_df_pop['GRID_ID_p'] = grid_ids_pop

In [ ]:
months = data_pop['month_year'].unique() 

# initialize a 24-hour zero array for each row in count_df_pop
# for each month
for month in months:
    count_df_pop[month] = [
        np.zeros(24, dtype=int) for _ in range(len(count_df_pop))
    ]

In [ ]:
previous_gid = None 
previous_month = None 

In [ ]:
for index, row in data_pop.iterrows():
    gid = row['GRID_ID_p']
    month_year = row['month_year']  
    deer_id = row['ind_loc_id']
    datetime_local = row['dt_local']
    hour = row['hour']
    
    date_range_start, date_range_end = calculate_date_range(month_year, timezone)

    gid_index = count_df_pop.index[count_df_pop['GRID_ID_p'] == gid].tolist()[0]  
    month_index = np.where(months == month_year)[0][0] 

    if (previous_gid != gid) and (
        date_range_start <= datetime_local <= date_range_end):
        count_df_pop.at[gid_index, month_year][hour] += 1
        previous_month = month_year

    elif (previous_gid == gid) and (previous_deer_id != deer_id) and (
        date_range_start <= datetime_local <= date_range_end):
        count_df_pop.at[gid_index, month_year][hour] += 1
        previous_month = month_year
        
    elif (previous_gid == gid) and (previous_deer_id == deer_id):
        if is_more_than_equal_2_hours(previous_datetime_local, datetime_local):
            start_time = previous_datetime_local + timedelta(hours=1) 
            end_time = datetime_local
        
            if previous_month != month_year:
                count_month = previous_month  
            else:
                count_month = month_year  

            while start_time <= end_time:
                start_hour = start_time.hour
                count_df_pop.at[gid_index, count_month][start_hour] += 1 
                start_time += timedelta(hours=1)
        else:
            count_df_pop.at[gid_index, previous_month][hour] += 1 
            
    previous_deer_id = deer_id
    previous_datetime_local = datetime_local 
    previous_gid = gid

In [ ]:
string_df = count_df_pop.copy()

# convert each list of hourly counts to a formatted string like '[0,1,2,...]'
for month in string_df.columns[1:]:  
    string_df[month] = string_df[month].apply(
        lambda x: '[' + ','.join(map(str, x)) + ']'
    )

In [ ]:
# rename columns to "Jan18_deer", "Feb18_deer", ...
string_df.rename(
    columns=lambda c: pd.to_datetime(c, format='%m/%Y').strftime('%b%y') +
    "_deer" if '/' in c else c,
    inplace=True
)

In [ ]:
feature_class4 = r"path\to\your\ArcGISProgeodatabase.gdb\parksmerged_tessellation"

count_dict_pop = string_df.set_index('GRID_ID_p').to_dict(orient='index')

In [ ]:
# check if fields exist, and add them if they don't; adds deer fields
for c_field in string_df.columns[1:]: 
    if c_field not in fields_list_pop:
        arcpy.AddField_management(feature_class4, c_field, "TEXT")

# update the feature class with the deer popularity by hour values
with arcpy.da.UpdateCursor(
    feature_class4, ['GRID_ID_p'] + list(string_df.columns[1:])
) as cursor:
    for row in cursor:
        gridid = row[0]  
        if gridid in count_dict_pop:
            for i, field in enumerate(string_df.columns[1:], start=1):
                row[i] = count_dict_pop[gridid][field]
            cursor.updateRow(row)

In [ ]:
# replace <Null> values
for c_field in string_df.columns[1:]:
    arcpy.CalculateField_management(
        feature_class4,
        c_field,
        "'[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]' "
        "if !{}! is None else !{}!".format(c_field, c_field),
        "PYTHON3"
    )

### Human Commercial Popularity by Hour

In [ ]:
arcpy.management.AddJoin(
    in_layer_or_view="human_popularity",
    in_field="placekey",
    join_table="poi_info",
    join_field="placekey",
    join_type="KEEP_COMMON",
    index_join_fields="NO_INDEX_JOIN_FIELDS",
    rebuild_index="NO_REBUILD_INDEX",
    join_operation=""
)

In [ ]:
arcpy.conversion.ExportTable(
    in_table="human_popularity",
    out_table=r"path\to\your\ArcGISProgeodatabase.gdb\human_popularity_ExportTable",
    where_clause="",
    use_field_alias_as_name="NOT_USE_ALIAS",
    field_mapping='placekey "placekey" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.placekey,0,7999;Jan18_hum "Jan18_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Jan18_hum,0,7999;Feb18_hum "Feb18_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Feb18_hum,0,7999;Mar18_hum "Mar18_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Mar18_hum,0,7999;Apr18_hum "Apr18_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Apr18_hum,0,7999;May18_hum "May18_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.May18_hum,0,7999;Jun18_hum "Jun18_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Jun18_hum,0,7999;Jul18_hum "Jul18_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Jul18_hum,0,7999;Aug18_hum "Aug18_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Aug18_hum,0,7999;Sep18_hum "Sep18_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Sep18_hum,0,7999;Oct18_hum "Oct18_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Oct18_hum,0,7999;Nov18_hum "Nov18_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Nov18_hum,0,7999;Dec18_hum "Dec18_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Dec18_hum,0,7999;Jan19_hum "Jan19_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Jan19_hum,0,7999;Feb19_hum "Feb19_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Feb19_hum,0,7999;Mar19_hum "Mar19_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Mar19_hum,0,7999;Apr19_hum "Apr19_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Apr19_hum,0,7999;May19_hum "May19_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.May19_hum,0,7999;Jun19_hum "Jun19_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Jun19_hum,0,7999;Jul19_hum "Jul19_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Jul19_hum,0,7999;Aug19_hum "Aug19_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Aug19_hum,0,7999;Sep19_hum "Sep19_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Sep19_hum,0,7999;Oct19_hum "Oct19_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Oct19_hum,0,7999;Nov19_hum "Nov19_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Nov19_hum,0,7999;Dec19_hum "Dec19_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Dec19_hum,0,7999;Jan20_hum "Jan20_hum" true true false 8000 Text 0 0,First,#,human_popularity,human_popularity.Jan20_hum,0,7999;OBJECTID "OBJECTID" false true false 4 Long 0 0,First,#,human_popularity,poi_info.OBJECTID,-1,-1;placekey "placekey" true true false 8000 Text 0 0,First,#,human_popularity,poi_info.placekey,0,7999;latitude "latitude" true true false 8 Double 0 0,First,#,human_popularity,poi_info.latitude,-1,-1;longitude "longitude" true true false 8 Double 0 0,First,#,human_popularity,poi_info.longitude,-1,-1;parent_placekey "parent_placekey" true true false 255 Text 0 0,First,#,human_popularity,poi_info.parent_placekey,0,254',
    sort_field=None
)

In [ ]:
fields_list_pop_hum = [
    field.name for field in arcpy.ListFields('human_popularity_ExportTable')
]

In [ ]:
data_pop_hum = pd.DataFrame.from_records(
    data=arcpy.da.SearchCursor(
        'human_popularity_ExportTable', fields_list_pop_hum
    ),
    columns=fields_list_pop_hum
)

In [ ]:
human_columns = [
    'Jan18_hum', 'Feb18_hum', 'Mar18_hum', 'Apr18_hum', 'May18_hum',
    'Jun18_hum', 'Jul18_hum', 'Aug18_hum', 'Sep18_hum', 'Oct18_hum',
    'Nov18_hum', 'Dec18_hum', 'Jan19_hum', 'Feb19_hum', 'Mar19_hum',
    'Apr19_hum', 'May19_hum', 'Jun19_hum', 'Jul19_hum', 'Aug19_hum',
    'Sep19_hum', 'Oct19_hum', 'Nov19_hum', 'Dec19_hum', 'Jan20_hum'
]

In [ ]:
flattened_data = []

for index, row in data_pop_hum.iterrows():
    for human_col in human_columns:
        month = human_col[:3]  # first three letters are always the month
        year = human_col[3:5]  # extract two-digit year

        pophuman_values = list(map(int, row[human_col][1:-1].split(',')))

        # create 24 records (one per hour)
        for hour, pophuman in enumerate(pophuman_values):
            flattened_data.append({
                'placekey': row['placekey'],
                'month_year': f"{month}{year}",
                'hour': hour,
                'pophuman': pophuman,
                'latitude': row['latitude'],
                'longitude': row['longitude']
            })

flattened_df = pd.DataFrame(flattened_data)

In [ ]:
arcpy.env.workspace = r"path\to\your\ArcGISProgeodatabase.gdb"

table_name = 'flattenedtable_pop'

# create a table based on flattened_df
arcpy.CreateTable_management(arcpy.env.workspace, table_name)

In [ ]:
# add fields to the table
arcpy.AddField_management(table_name, "placekey", "TEXT")
arcpy.AddField_management(table_name, "month_year", "TEXT")
arcpy.AddField_management(table_name, "hour", "SHORT")
arcpy.AddField_management(table_name, "pophuman", "LONG")
arcpy.AddField_management(table_name, "latitude", "DOUBLE")
arcpy.AddField_management(table_name, "longitude", "DOUBLE")

In [ ]:
# insert records
flattened_df_fields = [
    'placekey', 'month_year', 'hour', 'pophuman', 'latitude', 'longitude'
]

with arcpy.da.InsertCursor(table_name, flattened_df_fields) as cursor:
    for _, row in flattened_df.iterrows():
        cursor.insertRow((
            row['placekey'],
            row['month_year'],
            row['hour'],
            row['pophuman'],
            row['latitude'],
            row['longitude']
        ))

In [ ]:
arcpy.management.XYTableToPoint(
    in_table="flattenedtable_pop",
    out_feature_class=r"path\to\your\ArcGISProgeodatabase.gdb\flattenedtable_pop_XYTableToPoint",
    x_field="longitude",
    y_field="latitude",
    z_field=None,
    coordinate_system='GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]];-400 -400 1000000000;-100000 10000;-100000 10000;8.98315284119521E-09;0.001;0.001;IsHighPrecision'
)

In [ ]:
arcpy.analysis.SpatialJoin(
    target_features="flattenedtable_pop_XYTableToPoint",
    join_features="parksmerged_tessellation",
    out_feature_class=r"path\to\your\ArcGISProgeodatabase.gdb\flattenedtable_SpatialJoin",
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    field_mapping='placekey "placekey" true true false 255 Text 0 0,First,#,flattenedtable_pop_XYTableToPoint,placekey,0,254;month_year "month_year" true true false 255 Text 0 0,First,#,flattenedtable_pop_XYTableToPoint,month_year,0,254;hour "hour" true true false 2 Short 0 0,First,#,flattenedtable_pop_XYTableToPoint,hour,-1,-1;pophuman "pophuman" true true false 4 Long 0 0,First,#,flattenedtable_pop_XYTableToPoint,pophuman,-1,-1;latitude "latitude" true true false 8 Double 0 0,First,#,flattenedtable_pop_XYTableToPoint,latitude,-1,-1;longitude "longitude" true true false 8 Double 0 0,First,#,flattenedtable_pop_XYTableToPoint,longitude,-1,-1;GRID_ID_p "GRID_ID" true true false 12 Text 0 0,First,#,parksmerged_tessellation,GRID_ID_p,0,11',
    match_option="INTERSECT",
    search_radius=None,
    distance_field_name="",
    match_fields=None
)

In [ ]:
fields_list_pophum_fl = [
    field.name for field in arcpy.ListFields('flattenedtable_SpatialJoin')
]

In [ ]:
data_pophum_fl = pd.DataFrame.from_records(
    data=arcpy.da.SearchCursor(
        'flattenedtable_SpatialJoin', fields_list_pophum_fl
    ),
    columns=fields_list_pophum_fl
)

In [ ]:
# calculate the sums in the grid cells
data_hum_pop = (
    data_pophum_fl.groupby(['GRID_ID_p', 'month_year', 'hour'])
      .agg({'pophuman': 'sum'})
      .reset_index()
)

In [ ]:
hour_count_arrays = (
    data_hum_pop.groupby(['GRID_ID_p', 'month_year'])
         .apply(create_hour_count_array)
         .reset_index()
)

In [ ]:
hour_count_arrays.columns = ['GRID_ID_p', 'month_year', 'hour_counts']

hour_count_arrays['hour_counts'] = (
    hour_count_arrays['hour_counts'].apply(format_array)
)

In [ ]:
pivot_table = hour_count_arrays.pivot(
    index='GRID_ID_p',
    columns='month_year',
    values='hour_counts'
)

In [ ]:
pivot_table.columns = [f"{col}_hum" for col in pivot_table.columns] 

# sort columns by month_year
sorted_columns = ['GRID_ID_p'] + sorted(
    pivot_table.columns,
    key=lambda x: pd.to_datetime(x.replace('_hum', ''), format='%b%y')
)

In [ ]:
pivot_table = pivot_table.reset_index()  

pivot_table = pivot_table[
    ['GRID_ID_p'] + [col for col in sorted_columns if col != 'GRID_ID_p']
]

In [ ]:
feature_class5 = r"path\to\your\ArcGISProgeodatabase.gdb\parksmerged_tessellation"

count_dict_pop_hum = pivot_table.set_index('GRID_ID_p').to_dict(orient='index')

In [ ]:
# check if fields exist, and add them if they don't; adds human fields
for c_field in pivot_table.columns[1:]:
    if c_field not in fields_list_pophum_fl:
        arcpy.AddField_management(feature_class5, c_field, "TEXT")
        
# update the feature class with the deer popularity by hour values
with arcpy.da.UpdateCursor(
    feature_class5, ['GRID_ID_p'] + list(pivot_table.columns[1:])
) as cursor:
    for row in cursor:
        gridid = row[0]  
        if gridid in count_dict_pop_hum:
            for i, field in enumerate(pivot_table.columns[1:], start=1):
                row[i] = count_dict_pop_hum[gridid][field]
            cursor.updateRow(row)

In [ ]:
# replace <Null> values
for c_field in pivot_table.columns[1:]:
    arcpy.CalculateField_management(
        feature_class5,
        c_field,
        "'[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]' "
        "if !{}! is None else !{}!".format(c_field, c_field),
        "PYTHON3"
    )

*Note*: Human commercial popularity by hour values for the grid cell containing only Rockburn Branch Park were identified as outliers for the period August–October 2019. These values were replaced with corresponding data from August–October 2018.

In [ ]:
arcpy.conversion.ExportFeatures(
    in_features="parksmerged_tessellation",
    out_features=r"path\to\your\ArcGISProgeodatabase.gdb\popularity_201801_202001",
    where_clause="",
    use_field_alias_as_name="NOT_USE_ALIAS",
    field_mapping='GRID_ID_p "GRID_ID" true true false 12 Text 0 0,First,#,parksmerged_tessellation,GRID_ID_p,0,11;Shape_Length "Shape_Length" false true true 8 Double 0 0,First,#,parksmerged_tessellation,Shape_Length,-1,-1;Shape_Area "Shape_Area" false true true 8 Double 0 0,First,#,parksmerged_tessellation,Shape_Area,-1,-1;Jan18_deer "Jan18_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Jan18_deer,0,254;Feb18_deer "Feb18_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Feb18_deer,0,254;Mar18_deer "Mar18_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Mar18_deer,0,254;Apr18_deer "Apr18_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Apr18_deer,0,254;May18_deer "May18_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,May18_deer,0,254;Jun18_deer "Jun18_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Jun18_deer,0,254;Jul18_deer "Jul18_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Jul18_deer,0,254;Aug18_deer "Aug18_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Aug18_deer,0,254;Sep18_deer "Sep18_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Sep18_deer,0,254;Oct18_deer "Oct18_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Oct18_deer,0,254;Nov18_deer "Nov18_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Nov18_deer,0,254;Dec18_deer "Dec18_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Dec18_deer,0,254;Jan19_deer "Jan19_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Jan19_deer,0,254;Feb19_deer "Feb19_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Feb19_deer,0,254;Mar19_deer "Mar19_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Mar19_deer,0,254;Apr19_deer "Apr19_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Apr19_deer,0,254;May19_deer "May19_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,May19_deer,0,254;Jun19_deer "Jun19_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Jun19_deer,0,254;Jul19_deer "Jul19_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Jul19_deer,0,254;Aug19_deer "Aug19_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Aug19_deer,0,254;Sep19_deer "Sep19_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Sep19_deer,0,254;Oct19_deer "Oct19_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Oct19_deer,0,254;Nov19_deer "Nov19_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Nov19_deer,0,254;Dec19_deer "Dec19_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Dec19_deer,0,254;Jan20_deer "Jan20_deer" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Jan20_deer,0,254;Jan18_hum "Jan18_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Jan18_hum,0,254;Feb18_hum "Feb18_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Feb18_hum,0,254;Mar18_hum "Mar18_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Mar18_hum,0,254;Apr18_hum "Apr18_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Apr18_hum,0,254;May18_hum "May18_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,May18_hum,0,254;Jun18_hum "Jun18_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Jun18_hum,0,254;Jul18_hum "Jul18_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Jul18_hum,0,254;Aug18_hum "Aug18_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Aug18_hum,0,254;Sep18_hum "Sep18_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Sep18_hum,0,254;Oct18_hum "Oct18_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Oct18_hum,0,254;Nov18_hum "Nov18_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Nov18_hum,0,254;Dec18_hum "Dec18_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Dec18_hum,0,254;Jan19_hum "Jan19_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Jan19_hum,0,254;Feb19_hum "Feb19_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Feb19_hum,0,254;Mar19_hum "Mar19_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Mar19_hum,0,254;Apr19_hum "Apr19_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Apr19_hum,0,254;May19_hum "May19_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,May19_hum,0,254;Jun19_hum "Jun19_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Jun19_hum,0,254;Jul19_hum "Jul19_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Jul19_hum,0,254;Aug19_hum "Aug19_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Aug19_hum,0,254;Sep19_hum "Sep19_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Sep19_hum,0,254;Oct19_hum "Oct19_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Oct19_hum,0,254;Nov19_hum "Nov19_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Nov19_hum,0,254;Dec19_hum "Dec19_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Dec19_hum,0,254;Jan20_hum "Jan20_hum" true true false 255 Text 0 0,First,#,parksmerged_tessellation,Jan20_hum,0,254',
    sort_field=None
)